> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — covariance causal graph / mediation-ready structure

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
sim=vigor.simulate_genomic_dataset(n_samples=320,n_variants=1000,n_traits=8,seed=SEED,genotype_kwargs={"n_chromosomes":5,"generations":100},trait_kwargs=cfg)
X=sim.genotypes.to_pandas(); Y=sim.phenotypes.to_pandas()
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False

C=np.cov(Y.to_numpy(float),rowvar=False)
if vigor.pc_from_covariance is not None:
    graph=vigor.pc_from_covariance(C,n_eff=len(Y))
    print(type(graph))
else:
    print("PC optional dependency not installed; covariance matrix ready",C.shape)


In [ ]:
from vigor.causal import pc_from_data,genetic_covariance_mediation,anm_pairwise_direction,fit_multisource_covariance_sem
# Covariance mediation is a conditional association screen, not proof of a causal path.
fit=vigor.genetic_varcov_PSD(Ka,train,rank_G=4,rank_E=4,
    factor_structure_G="sparse_fa",factor_structure_E="sparse_fa",n_boot=0,progress=False,blas_threads=2)
mediation=genetic_covariance_mediation(np.asarray(fit["G"].to_pandas()),[0],outcomes=[1,2],E=np.asarray(fit["E"].to_pandas()),names=list(Y.columns))
print("mediation sources:",mediation.keys())
graph=pc_from_data(train.iloc[:,:4],min_pair_observed=20,max_cond_set=1)
print("PC:",type(graph).__name__)
rng=np.random.default_rng(42);cause=rng.normal(size=160);effect=cause**2+rng.normal(0,.2,160)
print("ANM demonstration:",anm_pairwise_direction(cause,effect,min_samples=100))
# An explicit candidate order demonstrates SEM fitting; it is not inferred causal truth.
sem=fit_multisource_covariance_sem({"G":np.asarray(fit["G"].to_pandas()),"E":np.asarray(fit["E"].to_pandas())},
    order=list(Y.columns),names=list(Y.columns),maxiter=100)
print("SEM coefficients:",np.asarray(sem["B"].to_pandas()).shape)
